In [ ]:
from openrouter import OpenRouter
from prompt_utils import *

import json
import random
import os

In [ ]:
# Configurations first
model = 'deepseek'
model_name = "deepseek/deepseek-v4.1-flash"

# Load the prompts
system_prompt = stance_emotion_system_prompt()
user_prompt = stance_emotion_user_prompt()

# Load the LLM engine
open_router_chat = OpenRouter(api_key="")

In [ ]:
# Load existing processed comments
processed_file = f'{model}_emotions_stance_processed'
processed_data = []

# Check if path exists
if os.path.exists(f'{processed_file}.json'):
    with open(f'{processed_file}.json', "r", encoding="utf-8") as f:
        processed_data = json.load(f)

print(f"Size of processed data: {len(processed_data)}")

In [ ]:
# Load the json file
filename = "all_comments.json"

new_data = []

with open(filename, "r", encoding="utf-8") as f:
    new_data = json.load(f)

print(f"Loaded: {len(new_data)} comments")

In [ ]:
# Perform subtraction

# Get the processed IDs
processed_ids = {item['row_id'] for item in processed_data}

# Remove
new_data = [item for item in new_data if item['row_id'] not in processed_ids]

print(f"After subtraction: {len(new_data)}")

In [ ]:
# Select a random comment
comment_record = random.choice(new_data)

print(comment_record['post_comment'])
print("\n\n")
print(comment_record['parent_text'])

In [ ]:
# Fill in the user prompt
user_prompt = user_prompt.format(
    parent_comment = comment_record['parent_text'] or '',
    post_comment = comment_record['post_comment']
)

print(user_prompt)

In [ ]:
# Pass the prompt to the LLM
res = open_router_chat.chat.send(
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
        "role":"user",
        "content": user_prompt
        }
    ],
    model = model_name
)

In [ ]:
# Parse the output
print(res.choices[0].message.content)

# Convert to dictionary/JSON
llm_output = json.loads(res.choices[0].message.content)

In [ ]:
# Add the JSON output to the comment record
comment_record['llm_output'] = llm_output

# Add to existing list
processed_data.append(comment_record)

In [ ]:
print(processed_data)

In [ ]:
# Save it in JSON
with open(f"{processed_file}.json", "w", encoding="utf-8") as f:
    json.dump(processed_data, f, ensure_ascii=False, indent=2)